### Basic chat bot

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    messages:Annotated[list, add_messages] #annoted list

graph_builder = StateGraph(State)
graph_builder

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

# os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY') or ''
print(os.environ['TAVILY_API_KEY'])
print(os.environ['GROQ_API_KEY'])

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm=ChatGroq(model='llama-3.1-8b-instant')
llm

In [ ]:
## Node functionality
def chatbot(state:State):
    return {'messages':[llm.invoke(state['messages'])]}

In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node('llmchatbot', chatbot)
graph_builder.add_edge(START, 'llmchatbot')
graph_builder.add_edge('llmchatbot', END)

graph = graph_builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage

result = graph.invoke({'messages':[HumanMessage(content='Hi')]})
result

In [ ]:
print(result)
print('\n')
print(result['messages'])
print('\n')
print(result['messages'][-1])
print('\n')
print(result['messages'][-1].content)

In [ ]:
for event in graph.stream({'messages': [HumanMessage(content='Hi, How are you?')]}):
    for value in event.values():
        print(value['messages'][-1].content)